# Chapter 7.2: Triton Programming — GPU Kernels in Python

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand why Triton exists** and how it compares to CUDA
2. **Write Triton kernels** using the block-based programming model
3. **Use `tl.load` and `tl.store`** for memory operations with masking
4. **Implement core operations**: vector add, matmul, softmax
5. **Apply auto-tuning** with `@triton.autotune` for optimal performance
6. **Fuse operations** to reduce memory traffic
7. **Benchmark Triton vs CUDA** and understand the trade-offs

## Prerequisites

- Chapter 7.1 (CUDA Primer)
- `triton` package installed (`pip install triton`)
- NVIDIA GPU with compute capability >= 7.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.2_triton.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.2_triton.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Triton?

### 1.1 The Problem with CUDA

CUDA is powerful but has a steep learning curve:

| CUDA Challenge | Impact |
|---------------|--------|
| Manual thread indexing | Easy to get wrong, hard to debug |
| Manual shared memory management | Bank conflicts, synchronization bugs |
| Manual memory coalescing | Non-obvious performance cliffs |
| Complex launch configuration | Grid/block dims affect performance |
| C++ compilation required | Slow iteration, complex build systems |

### 1.2 Triton's Approach

Triton raises the abstraction level from **individual threads** to **blocks of data**:

```
CUDA: "Thread 42 loads element A[42], multiplies by B[42], stores to C[42]"
Triton: "This program loads BLOCK_SIZE elements, operates on them, stores results"
```

**What Triton handles automatically:**
- Shared memory allocation and management
- Memory coalescing
- Thread-level parallelism within a block
- Warp-level optimizations
- Register allocation

**What you still control:**
- Block size (BLOCK_SIZE)
- Data layout and access patterns
- Algorithm design (tiling, fusion)
- Launch grid dimensions

### 1.3 Triton in the LLM Ecosystem

Triton is used extensively in production LLM systems:

| Project | Triton Usage |
|---------|-------------|
| **vLLM** | PagedAttention, fused ops, quantized kernels |
| **SGLang** | Attention kernels, fused normalization |
| **PyTorch 2.0** | `torch.compile` generates Triton kernels |
| **FlashAttention-Triton** | Pure Triton implementation of FlashAttention |
| **xformers** | Memory-efficient attention |
| **Unsloth** | Training optimizations |

In [ ]:
# Verify Triton installation and GPU
import torch

try:
    import triton
    import triton.language as tl
    print(f"Triton version: {triton.__version__}")
except ImportError:
    print("Triton not installed. Install with: pip install triton")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Compute Capability: {torch.cuda.get_device_capability()}")
else:
    print("No CUDA GPU available.")

## 2. Triton Programming Model

### 2.1 Programs vs Threads

In CUDA, you think about individual **threads**. In Triton, you think about **programs** that operate on **blocks** of data.

```python
# CUDA thinking:
# "Thread tid loads A[tid], adds B[tid], stores to C[tid]"

# Triton thinking:
# "Program pid loads A[pid*BLOCK:(pid+1)*BLOCK], adds B[same], stores to C[same]"
```

Each Triton program maps to a **CUDA thread block** under the hood. The compiler handles all intra-block parallelism.

### 2.2 Key Concepts

| Concept | Description |
|---------|------------|
| `@triton.jit` | Decorator that JIT-compiles a function to GPU code |
| `tl.program_id(axis)` | Returns the program index (like `blockIdx` in CUDA) |
| `tl.arange(0, N)` | Creates a range [0, N) — always a power of 2 |
| `tl.load(ptr, mask)` | Loads a block of data from memory |
| `tl.store(ptr, val, mask)` | Stores a block of data to memory |
| `tl.constexpr` | Compile-time constant (like template parameters in C++) |

In [ ]:
import torch
import triton
import triton.language as tl

# ============================================================
# Vector Addition — The "Hello World" of Triton
# ============================================================

@triton.jit
def vector_add_kernel(
    a_ptr,          # Pointer to first input vector
    b_ptr,          # Pointer to second input vector  
    c_ptr,          # Pointer to output vector
    N,              # Number of elements
    BLOCK_SIZE: tl.constexpr,  # Block size — must be known at compile time
):
    """
    Each program instance processes BLOCK_SIZE elements.
    Program i processes elements [i*BLOCK_SIZE, (i+1)*BLOCK_SIZE).
    """
    # Step 1: Which block of data does this program handle?
    pid = tl.program_id(0)  # 1D launch: program index along axis 0
    
    # Step 2: Compute the offsets for this block
    # offsets = [pid*BLOCK_SIZE, pid*BLOCK_SIZE+1, ..., pid*BLOCK_SIZE+BLOCK_SIZE-1]
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    
    # Step 3: Create a mask for boundary handling
    # If N is not divisible by BLOCK_SIZE, some elements in the last block are invalid
    mask = offsets < N
    
    # Step 4: Load data from global memory
    # tl.load handles the entire block at once — no manual thread indexing!
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    
    # Step 5: Compute (operates on entire block)
    c = a + b
    
    # Step 6: Store result back to global memory
    tl.store(c_ptr + offsets, c, mask=mask)


# Host code: prepare data and launch kernel
def vector_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Wrapper function that launches the Triton kernel."""
    assert a.shape == b.shape
    assert a.is_cuda and b.is_cuda
    
    c = torch.empty_like(a)
    N = a.numel()
    
    # Launch configuration: how many programs to launch
    BLOCK_SIZE = 1024
    grid = ((N + BLOCK_SIZE - 1) // BLOCK_SIZE,)  # 1D grid
    
    # Launch!
    vector_add_kernel[grid](a, b, c, N, BLOCK_SIZE=BLOCK_SIZE)
    
    return c


# Test
if torch.cuda.is_available():
    N = 1_000_000
    a = torch.randn(N, device='cuda')
    b = torch.randn(N, device='cuda')
    
    c = vector_add(a, b)
    c_ref = a + b
    
    print(f"Max error: {(c - c_ref).abs().max().item():.2e}")
    print("PASSED!" if torch.allclose(c, c_ref) else "FAILED!")

## 3. Memory Operations: `tl.load` and `tl.store`

### 3.1 Pointer Arithmetic

Triton uses **pointer arithmetic** similar to C, but operating on blocks:

```python
# ptr + offsets creates a block of pointers
# If ptr = 0x1000 and offsets = [0, 1, 2, 3]
# Then ptr + offsets = [0x1000, 0x1004, 0x1008, 0x100C]  (float32 = 4 bytes)
```

### 3.2 Masking

Masks prevent out-of-bounds memory access:

```python
mask = offsets < N  # Boolean tensor: True for valid elements
data = tl.load(ptr + offsets, mask=mask, other=0.0)  # Load 0.0 for masked elements
```

### 3.3 Multi-dimensional Access

For 2D data (matrices), use broadcasting to create 2D offset patterns:

```python
# Row offsets: shape (BLOCK_M, 1)
row_offsets = tl.arange(0, BLOCK_M)[:, None]
# Column offsets: shape (1, BLOCK_N)
col_offsets = tl.arange(0, BLOCK_N)[None, :]
# 2D offsets: shape (BLOCK_M, BLOCK_N)
offsets = row_offsets * stride_row + col_offsets * stride_col
```

In [ ]:
# Demonstrate 2D memory access patterns
import torch
import triton
import triton.language as tl

@triton.jit
def transpose_kernel(
    input_ptr,
    output_ptr,
    M, N,
    input_stride_m,   # Stride along rows of input
    input_stride_n,   # Stride along cols of input  
    output_stride_m,  # Stride along rows of output
    output_stride_n,  # Stride along cols of output
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """
    Transpose a matrix: output[j, i] = input[i, j]
    
    Each program handles a BLOCK_M × BLOCK_N tile.
    """
    # Which tile does this program handle?
    pid_m = tl.program_id(0)  # Tile row index
    pid_n = tl.program_id(1)  # Tile column index
    
    # Compute offsets for the tile in the input matrix
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    
    # 2D indexing using broadcasting
    # input_offsets shape: (BLOCK_M, BLOCK_N)
    input_offsets = row_offsets[:, None] * input_stride_m + col_offsets[None, :] * input_stride_n
    
    # Mask for boundary
    mask = (row_offsets[:, None] < M) & (col_offsets[None, :] < N)
    
    # Load the tile
    tile = tl.load(input_ptr + input_offsets, mask=mask)
    
    # Compute output offsets (transposed: swap row and col)
    output_offsets = col_offsets[:, None] * output_stride_m + row_offsets[None, :] * output_stride_n
    output_mask = (col_offsets[:, None] < N) & (row_offsets[None, :] < M)
    
    # Store transposed tile
    # Note: tl.trans() transposes the 2D block
    tl.store(output_ptr + output_offsets, tl.trans(tile), mask=output_mask)


def transpose(x: torch.Tensor) -> torch.Tensor:
    M, N = x.shape
    out = torch.empty(N, M, device=x.device, dtype=x.dtype)
    
    BLOCK_M, BLOCK_N = 32, 32
    grid = ((M + BLOCK_M - 1) // BLOCK_M, (N + BLOCK_N - 1) // BLOCK_N)
    
    transpose_kernel[grid](
        x, out,
        M, N,
        x.stride(0), x.stride(1),
        out.stride(0), out.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )
    return out


if torch.cuda.is_available():
    x = torch.randn(128, 64, device='cuda')
    y = transpose(x)
    y_ref = x.T.contiguous()
    
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {y.shape}")
    print(f"Max error: {(y - y_ref).abs().max().item():.2e}")
    print("PASSED!" if torch.allclose(y, y_ref) else "FAILED!")

## 4. Matrix Multiplication in Triton

### 4.1 The Tiling Strategy

Matrix multiplication `C = A @ B` where `A` is `(M, K)` and `B` is `(K, N)` uses the same tiling approach as CUDA, but expressed more naturally:

```
For each output tile C[m:m+BLOCK_M, n:n+BLOCK_N]:
    accumulator = zeros(BLOCK_M, BLOCK_N)
    for k in range(0, K, BLOCK_K):
        a_tile = A[m:m+BLOCK_M, k:k+BLOCK_K]   # Load tile from A
        b_tile = B[k:k+BLOCK_K, n:n+BLOCK_N]   # Load tile from B
        accumulator += a_tile @ b_tile            # Tile-level matmul
    C[m:m+BLOCK_M, n:n+BLOCK_N] = accumulator
```

### 4.2 The `tl.dot` Operation

`tl.dot(a, b)` computes a matrix multiply of two 2D blocks. Under the hood, this maps to **tensor core** operations on supported hardware.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    # Pointers to matrices
    a_ptr, b_ptr, c_ptr,
    # Matrix dimensions
    M, N, K,
    # Strides (number of elements to skip to move one row/col)
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # Block sizes (compile-time constants)
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    Compute C = A @ B using tiled matrix multiplication.
    
    Each program computes a BLOCK_M × BLOCK_N tile of C.
    """
    # ---- Step 1: Determine which tile of C this program computes ----
    pid_m = tl.program_id(0)  # Row tile index
    pid_n = tl.program_id(1)  # Column tile index
    
    # ---- Step 2: Create offset arrays for the tile ----
    # Row offsets for A and C
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # shape: (BLOCK_M,)
    # Column offsets for B and C
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # shape: (BLOCK_N,)
    # K-dimension offsets (start at 0, advance by BLOCK_K each iteration)
    rk = tl.arange(0, BLOCK_K)  # shape: (BLOCK_K,)
    
    # ---- Step 3: Create pointers to first tiles of A and B ----
    # A tile: rows rm, columns rk
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    # B tile: rows rk, columns rn  
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    
    # ---- Step 4: Accumulate over K tiles ----
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        # Load tiles with boundary masking
        a_mask = (rm[:, None] < M) & ((k + rk[None, :]) < K)
        b_mask = ((k + rk[:, None]) < K) & (rn[None, :] < N)
        
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        
        # Tile matrix multiply — maps to tensor cores!
        acc += tl.dot(a, b)
        
        # Advance pointers to next K tile
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    # ---- Step 5: Store the result tile ----
    c_offsets = rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptr + c_offsets, acc, mask=c_mask)


def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Inner dimensions must match"
    
    c = torch.empty(M, N, device=a.device, dtype=torch.float32)
    
    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
    grid = ((M + BLOCK_M - 1) // BLOCK_M, (N + BLOCK_N - 1) // BLOCK_N)
    
    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c


if torch.cuda.is_available():
    M, N, K = 512, 512, 512
    a = torch.randn(M, K, device='cuda', dtype=torch.float32)
    b = torch.randn(K, N, device='cuda', dtype=torch.float32)
    
    c = matmul(a, b)
    c_ref = torch.mm(a, b)
    
    print(f"Matrix multiply: ({M}, {K}) @ ({K}, {N})")
    print(f"Max error: {(c - c_ref).abs().max().item():.6f}")
    print("PASSED!" if torch.allclose(c, c_ref, atol=1e-2) else "FAILED!")

## 5. Softmax in Triton

### 5.1 Why Softmax is Important

Softmax is used in **every attention layer** of every transformer:

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

The subtraction of `max(x)` is crucial for **numerical stability** (prevents overflow in exp).

### 5.2 Softmax Algorithm

1. Find `m = max(x)` over the row
2. Compute `e_i = exp(x_i - m)` for each element
3. Find `s = sum(e_i)` over the row
4. Divide: `y_i = e_i / s`

A naive implementation reads the row **3 times** from global memory. A fused kernel reads it **once**.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def softmax_kernel(
    input_ptr,
    output_ptr,
    n_cols,
    input_stride,   # Stride between rows
    output_stride,
    BLOCK_SIZE: tl.constexpr,  # Must be >= n_cols (one program per row)
):
    """
    Compute row-wise softmax.
    
    Each program handles ONE ROW of the input.
    The entire row is loaded into SRAM (shared memory/registers),
    so we only read from global memory ONCE.
    """
    # Which row does this program handle?
    row_idx = tl.program_id(0)
    
    # Pointer to this row's data
    row_start = input_ptr + row_idx * input_stride
    
    # Column offsets
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols
    
    # Step 1: Load the entire row from global memory (ONE read)
    row = tl.load(row_start + col_offsets, mask=mask, other=-float('inf'))
    
    # Step 2: Compute max for numerical stability
    row_max = tl.max(row, axis=0)  # Scalar: max over the block
    
    # Step 3: Compute exp(x - max)
    numerator = tl.exp(row - row_max)
    
    # Step 4: Compute sum of exponentials
    denominator = tl.sum(numerator, axis=0)  # Scalar: sum over the block
    
    # Step 5: Divide
    result = numerator / denominator
    
    # Step 6: Store result (ONE write)
    out_start = output_ptr + row_idx * output_stride
    tl.store(out_start + col_offsets, result, mask=mask)


def softmax(x: torch.Tensor) -> torch.Tensor:
    """Row-wise softmax using Triton."""
    n_rows, n_cols = x.shape
    
    # BLOCK_SIZE must be a power of 2 >= n_cols
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    
    out = torch.empty_like(x)
    
    # One program per row
    grid = (n_rows,)
    
    softmax_kernel[grid](
        x, out,
        n_cols,
        x.stride(0), out.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out


if torch.cuda.is_available():
    x = torch.randn(128, 256, device='cuda')
    
    y = softmax(x)
    y_ref = torch.softmax(x, dim=-1)
    
    print(f"Input shape: {x.shape}")
    print(f"Max error: {(y - y_ref).abs().max().item():.2e}")
    print(f"Sum per row (should be 1.0): {y.sum(dim=-1)[:5]}")
    print("PASSED!" if torch.allclose(y, y_ref, atol=1e-5) else "FAILED!")

## 6. Auto-Tuning with `@triton.autotune`

### 6.1 Why Auto-Tune?

The optimal block size depends on many factors:
- Matrix dimensions
- GPU architecture (A100 vs H100)
- Memory access patterns
- Register pressure

Instead of guessing, Triton can **automatically try multiple configurations** and pick the fastest one:

```python
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 128, 'BLOCK_K': 64}, num_warps=8),
        # ... more configs
    ],
    key=['M', 'N', 'K'],  # Re-tune when these change
)
```

In [ ]:
import torch
import triton
import triton.language as tl

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64,  'BLOCK_K': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64,  'BLOCK_K': 64}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32,  'BLOCK_N': 32,  'BLOCK_K': 32}, num_warps=2, num_stages=2),
    ],
    key=['M', 'N', 'K'],
)
@triton.jit
def matmul_autotuned_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """Same matmul kernel but with autotuning."""
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        a_mask = (rm[:, None] < M) & ((k + rk[None, :]) < K)
        b_mask = ((k + rk[:, None]) < K) & (rn[None, :] < N)
        
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc += tl.dot(a, b)
        
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    c_offsets = rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptr + c_offsets, acc, mask=c_mask)


def matmul_autotuned(a, b):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty(M, N, device=a.device, dtype=torch.float32)
    
    grid = lambda meta: (
        (M + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],
        (N + meta['BLOCK_N'] - 1) // meta['BLOCK_N'],
    )
    
    matmul_autotuned_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c


if torch.cuda.is_available():
    M, N, K = 1024, 1024, 1024
    a = torch.randn(M, K, device='cuda', dtype=torch.float32)
    b = torch.randn(K, N, device='cuda', dtype=torch.float32)
    
    # First call triggers autotuning
    print("Running autotuning (first call)...")
    c = matmul_autotuned(a, b)
    c_ref = torch.mm(a, b)
    print(f"Max error: {(c - c_ref).abs().max().item():.6f}")
    print("Autotuning complete. Subsequent calls use the best config.")

## 7. Fused Operations in Triton

### 7.1 Why Fusion?

Consider a sequence of operations: `Y = dropout(softmax(X @ W))`

**Without fusion** (3 separate kernels):
```
Kernel 1: tmp1 = X @ W      → Write tmp1 to HBM (2 TB/s bottleneck)
Kernel 2: tmp2 = softmax(tmp1) → Read tmp1, Write tmp2 to HBM
Kernel 3: Y = dropout(tmp2)    → Read tmp2, Write Y to HBM
Total: 6 global memory operations
```

**With fusion** (1 kernel):
```
Kernel 1: Y = dropout(softmax(X @ W))  → Only 2 global memory operations
Total: 2 global memory operations → 3x less memory traffic!
```

### 7.2 Triton Makes Fusion Easy

In Triton, you simply write the fused logic as a single kernel. The compiler handles the rest.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_softmax_dropout_kernel(
    input_ptr,
    output_ptr,
    seed,          # Random seed for dropout
    n_rows, n_cols,
    input_stride,
    output_stride,
    p,             # Dropout probability
    BLOCK_SIZE: tl.constexpr,
):
    """
    Fused softmax + dropout in a single kernel.
    
    This saves one round-trip to global memory compared to
    doing softmax and dropout separately.
    """
    row_idx = tl.program_id(0)
    
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols
    
    # Load row
    row_ptr = input_ptr + row_idx * input_stride
    row = tl.load(row_ptr + col_offsets, mask=mask, other=-float('inf'))
    
    # Softmax
    row_max = tl.max(row, axis=0)
    numerator = tl.exp(row - row_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_out = numerator / denominator
    
    # Dropout (generate random mask)
    # Triton provides tl.rand for generating random numbers
    random = tl.rand(seed, row_idx * BLOCK_SIZE + col_offsets)
    dropout_mask = random > p
    scale = 1.0 / (1.0 - p)  # Scale to maintain expected values
    result = tl.where(dropout_mask, softmax_out * scale, 0.0)
    
    # Store result
    out_ptr = output_ptr + row_idx * output_stride
    tl.store(out_ptr + col_offsets, result, mask=mask)


def fused_softmax_dropout(x, p=0.1):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    seed = torch.randint(0, 2**31, (1,)).item()
    
    fused_softmax_dropout_kernel[(n_rows,)](
        x, out, seed,
        n_rows, n_cols,
        x.stride(0), out.stride(0),
        p,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out


if torch.cuda.is_available():
    x = torch.randn(32, 128, device='cuda')
    y = fused_softmax_dropout(x, p=0.1)
    
    # Check that some elements are zero (dropped)
    zero_frac = (y == 0).float().mean().item()
    print(f"Fraction of zeros (dropout p=0.1): {zero_frac:.3f}")
    print(f"Expected ~0.10, got {zero_frac:.3f}")
    
    # Non-zero elements should still form valid softmax distributions
    # (after accounting for scaling)
    print(f"\nOutput shape: {y.shape}")
    print(f"Sample row sums (with dropout, expect ~1.0): {y.sum(dim=-1)[:5]}")

## 8. Benchmarking Triton vs PyTorch

Let's systematically benchmark our Triton kernels against PyTorch's built-in operations.

In [ ]:
import torch
import triton
import time

def benchmark_fn(fn, *args, warmup=10, n_iter=100, label=""):
    """Benchmark a function with proper GPU synchronization."""
    # Warmup
    for _ in range(warmup):
        fn(*args)
    
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        fn(*args)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_iter
    
    return elapsed


if torch.cuda.is_available():
    print("=" * 80)
    print("SOFTMAX BENCHMARK")
    print("=" * 80)
    
    for n_cols in [128, 256, 512, 1024, 2048, 4096]:
        n_rows = 1024
        x = torch.randn(n_rows, n_cols, device='cuda')
        
        triton_time = benchmark_fn(softmax, x)
        pytorch_time = benchmark_fn(lambda x: torch.softmax(x, dim=-1), x)
        
        # Bandwidth calculation
        bytes_accessed = 2 * n_rows * n_cols * 4  # Read + write, float32
        triton_bw = bytes_accessed / triton_time / 1e9
        pytorch_bw = bytes_accessed / pytorch_time / 1e9
        
        speedup = pytorch_time / triton_time
        print(f"Cols={n_cols:5d} | Triton: {triton_time*1e6:7.1f}µs ({triton_bw:6.1f} GB/s) | "
              f"PyTorch: {pytorch_time*1e6:7.1f}µs ({pytorch_bw:6.1f} GB/s) | "
              f"Speedup: {speedup:.2f}x")
    
    print()
    print("=" * 80)
    print("VECTOR ADD BENCHMARK")
    print("=" * 80)
    
    for N in [1024, 1024*64, 1024*1024, 1024*1024*16]:
        a = torch.randn(N, device='cuda')
        b = torch.randn(N, device='cuda')
        
        triton_time = benchmark_fn(vector_add, a, b)
        pytorch_time = benchmark_fn(lambda a, b: a + b, a, b)
        
        bytes_accessed = 3 * N * 4  # 2 reads + 1 write
        triton_bw = bytes_accessed / triton_time / 1e9
        pytorch_bw = bytes_accessed / pytorch_time / 1e9
        
        speedup = pytorch_time / triton_time
        print(f"N={N:>10,} | Triton: {triton_time*1e6:7.1f}µs ({triton_bw:6.1f} GB/s) | "
              f"PyTorch: {pytorch_time*1e6:7.1f}µs ({pytorch_bw:6.1f} GB/s) | "
              f"Speedup: {speedup:.2f}x")
else:
    print("GPU not available for benchmarking.")

## 9. Advanced Triton Patterns

### 9.1 Reduction Across Blocks

Some operations (like RMSNorm) require reducing across an entire row that may be larger than one block. The pattern is:
1. Each program computes a partial reduction
2. Use `tl.atomic_add` to accumulate across programs

### 9.2 Indirect Addressing (PagedAttention)

PagedAttention uses a block table to look up where KV cache blocks are stored. In Triton, this is done with `tl.load` for the table lookup followed by another `tl.load` for the actual data.

### 9.3 Mixed Precision

Triton supports mixed precision naturally:

```python
# Load in FP16, accumulate in FP32
a = tl.load(a_ptr + offsets).to(tl.float32)  # FP16 → FP32
acc += tl.dot(a, b)  # FP32 accumulation
result = acc.to(tl.float16)  # FP32 → FP16 for storage
```

In [ ]:
# RMSNorm in Triton — a key operation in LLaMA-style models
import torch
import triton
import triton.language as tl

@triton.jit
def rmsnorm_kernel(
    input_ptr,
    weight_ptr,
    output_ptr,
    n_cols,
    eps,
    input_stride,
    output_stride,
    BLOCK_SIZE: tl.constexpr,
):
    """
    RMSNorm: y = x * weight / sqrt(mean(x^2) + eps)
    
    Each program handles one row.
    This is a fused kernel: norm computation + scaling in one pass.
    """
    row_idx = tl.program_id(0)
    
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols
    
    # Load input row
    row_ptr = input_ptr + row_idx * input_stride
    x = tl.load(row_ptr + col_offsets, mask=mask, other=0.0).to(tl.float32)
    
    # Compute RMS: sqrt(mean(x^2) + eps)
    x_sq = x * x
    mean_sq = tl.sum(x_sq, axis=0) / n_cols
    rms = tl.rsqrt(mean_sq + eps)  # 1 / sqrt(mean_sq + eps)
    
    # Normalize
    x_norm = x * rms
    
    # Load weight and apply
    weight = tl.load(weight_ptr + col_offsets, mask=mask)
    result = x_norm * weight
    
    # Store
    out_ptr = output_ptr + row_idx * output_stride
    tl.store(out_ptr + col_offsets, result, mask=mask)


def rmsnorm(x, weight, eps=1e-6):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    
    rmsnorm_kernel[(n_rows,)](
        x, weight, out,
        n_cols, eps,
        x.stride(0), out.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out


if torch.cuda.is_available():
    # Test RMSNorm
    hidden_size = 4096
    batch_size = 32
    
    x = torch.randn(batch_size, hidden_size, device='cuda')
    weight = torch.ones(hidden_size, device='cuda')
    
    # Triton version
    y = rmsnorm(x, weight)
    
    # Reference implementation
    rms = torch.sqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + 1e-6)
    y_ref = (x.float() / rms * weight).to(x.dtype)
    
    print(f"RMSNorm shape: ({batch_size}, {hidden_size})")
    print(f"Max error: {(y - y_ref).abs().max().item():.2e}")
    print("PASSED!" if torch.allclose(y, y_ref, atol=1e-4) else "FAILED!")

## 10. Triton vs CUDA: When to Use Which?

### Decision Matrix

| Factor | Choose Triton | Choose CUDA |
|--------|-------------|-------------|
| **Development speed** | Fast iteration in Python | Slower C++ development |
| **Performance** | 80-95% of hand-tuned CUDA | Maximum possible performance |
| **Debugging** | Print statements, Python errors | cuda-gdb, Nsight, printf |
| **Tensor cores** | Automatic via `tl.dot` | Manual WMMA/MMA intrinsics |
| **Shared memory** | Automatic management | Full manual control |
| **Warp-level ops** | Limited (`tl.sum`, `tl.max`) | Full shuffle/vote/ballot |
| **Complex control flow** | Limited | Full support |
| **Multi-GPU** | Via PyTorch integration | Direct NCCL/CUDA APIs |

### In Practice

- **Start with Triton** for prototyping and most production kernels
- **Switch to CUDA** when you need:
  - Maximum performance for a bottleneck kernel
  - Complex warp-level programming
  - Fine-grained shared memory control
  - Integration with CUTLASS or other CUDA libraries

In [ ]:
# Summary comparison: Lines of code for common operations
comparison = {
    "Vector Add": {"Triton": 15, "CUDA": 25},
    "Softmax": {"Triton": 25, "CUDA": 80},
    "Matmul": {"Triton": 40, "CUDA": 120},
    "RMSNorm": {"Triton": 25, "CUDA": 70},
    "Fused Softmax+Dropout": {"Triton": 30, "CUDA": 100},
}

print(f"{'Operation':<25} {'Triton LOC':>12} {'CUDA LOC':>12} {'Ratio':>8}")
print("-" * 60)
for op, locs in comparison.items():
    ratio = locs['CUDA'] / locs['Triton']
    print(f"{op:<25} {locs['Triton']:>12} {locs['CUDA']:>12} {ratio:>7.1f}x")

print("\n→ Triton requires 2-4x fewer lines of code for equivalent functionality.")
print("→ The gap increases for more complex operations.")

## 11. Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Programming model** | Block-level operations, not thread-level |
| **Memory ops** | `tl.load`/`tl.store` with masking for safety |
| **Computation** | Standard Python operators + `tl.dot` for matmul |
| **Auto-tuning** | `@triton.autotune` finds optimal block sizes automatically |
| **Fusion** | Natural — just write combined logic in one kernel |
| **Performance** | 80-95% of hand-tuned CUDA with much less effort |

### Next Steps

In Chapter 7.3, we'll use Triton to implement **PagedAttention** — the key innovation in vLLM that enables efficient KV cache memory management.

## Exercises

### Exercise 1: Layer Normalization

Implement LayerNorm in Triton:
$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

Where $\mu$ and $\sigma^2$ are the mean and variance computed over each row.

### Exercise 2: GELU Activation

Implement the GELU activation function used in GPT-style models:
$$\text{GELU}(x) = 0.5 \cdot x \cdot (1 + \tanh(\sqrt{2/\pi} \cdot (x + 0.044715 \cdot x^3)))$$

### Exercise 3: Fused Attention Score

Implement a kernel that computes the attention scores for a single head:
1. Compute `S = Q @ K^T / sqrt(d_k)`
2. Apply causal mask
3. Compute `softmax(S)`

Do this in a single fused kernel.

In [ ]:
# Exercise 1 Skeleton: LayerNorm

@triton.jit
def layernorm_kernel(
    input_ptr, gamma_ptr, beta_ptr, output_ptr,
    n_cols, eps,
    input_stride, output_stride,
    BLOCK_SIZE: tl.constexpr,
):
    row_idx = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols
    
    # TODO: Load the row
    # TODO: Compute mean
    # TODO: Compute variance
    # TODO: Normalize
    # TODO: Apply gamma and beta
    # TODO: Store result
    pass

print("Exercise 1: Implement LayerNorm in Triton.")
print("Fill in the TODO sections above.")
print("Test against torch.nn.LayerNorm.")

In [ ]:
# Exercise 2 Skeleton: GELU

@triton.jit
def gelu_kernel(
    input_ptr, output_ptr, N,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    
    x = tl.load(input_ptr + offsets, mask=mask)
    
    # TODO: Implement GELU
    # GELU(x) = 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    # Hint: Use tl.math.tanh and tl.math.sqrt
    
    # result = ...
    # tl.store(output_ptr + offsets, result, mask=mask)
    pass

print("Exercise 2: Implement GELU activation in Triton.")
print("Test against torch.nn.functional.gelu(x, approximate='tanh').")

In [ ]:
# Exercise 3 Skeleton: Fused Attention Score

print("Exercise 3: Implement fused attention scores.")
print()
print("The kernel should:")
print("1. Compute S = Q @ K^T / sqrt(d_k) for a single head")
print("2. Apply causal mask: S[i,j] = -inf if j > i")
print("3. Compute softmax over the last dimension")
print()
print("Hints:")
print("- Each program handles one query position (one row of S)")
print("- Load the query vector and iterate over K tiles")
print("- Use online softmax for numerical stability")
print("- This is a simplified version of FlashAttention (Chapter 7.4)")